In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE DATABASE IF NOT EXISTS gold")


DataFrame[]

## 1. gold.dim_movies

A dimensão de filmes usa uma chave substituta `sk_movie_id` e mantém os principais metadados descritivos.


In [0]:
df_info = spark.table("silver.tb_info_filmes")

janela_movie = Window.orderBy("id_filme")

df_dim_movies = (
    df_info
    .withColumn(
        "sk_movie_id",
        F.row_number().over(janela_movie).cast("bigint")
    )
    .select(
        "sk_movie_id",
        F.col("id_filme").cast("string"),
        F.col("titulo").cast("string"),
        F.col("data_lancamento").cast("date"),
        F.col("ano_lancamento").cast("int"),
        F.col("duracao_minutos").cast("int"),
        F.col("idioma_original").cast("string"),
        F.col("status_filme").cast("string"),
        F.col("sinopse").cast("string")
    )
)

(
    df_dim_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_movies")
)

display(df_dim_movies.limit(20))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,1000004,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
2,1000005,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs."
3,1000007,KYLE BROWNRIGG: INTRODUCING LYLE,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
4,1000011,Worth Your Weight in Gold,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
5,1000014,On va manquer !,2018-05-15,2018,0,fr,Lançado,null
6,1000030,58 Hours: The Baby Jessica Story,2021-07-31,2021,0,es,Lançado,null
7,1000054,One Hundred Years and Hope,2022-06-18,2022,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope."
8,1000058,Homecoming,2023-07-12,2023,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances."
9,1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,2016,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!"
10,1000073,A Chance To Win,2023-05-03,2023,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever."


## 2. gold.fact_movies_performance

Grão: um registro por filme lançado. A fato consolida métricas financeiras e de engajamento sem multiplicar o grão nos joins.


In [0]:
df_movies_lancados = (
    spark.table("gold.dim_movies")
    .filter(F.col("status_filme") == "Lançado")
)

df_fin = (
    spark.table("silver.tb_financeiro_filmes")
    .dropDuplicates(["id_filme"])
)

df_metricas = (
    spark.table("silver.tb_metricas_engajamento")
    .dropDuplicates(["id_filme"])
)

df_fact_movies = (
    df_movies_lancados
    .select("sk_movie_id", "id_filme")
    .join(df_fin, on="id_filme", how="left")
    .join(df_metricas, on="id_filme", how="left")
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int")
    )
)

duplicados_fact = (
    df_fact_movies
    .groupBy("sk_movie_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicados_fact != 0:
    raise Exception(f"A fato possui {duplicados_fact} sk_movie_id duplicadas.")

(
    df_fact_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fact_movies_performance")
)

print("Total de linhas na fato:", df_fact_movies.count())
print("SKs duplicadas:", duplicados_fact)
display(df_fact_movies.limit(20))


Total de linhas na fato: 96463
SKs duplicadas: 0


sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1,null,null,null,null,null,null,1.132,0.0,0,6.8,27
2,null,null,null,null,null,null,0.6,0.0,0,null,40
3,null,null,null,null,null,null,0.6,0.0,0,4.7,10
4,null,null,null,null,null,null,1.169,0.0,0,4.8,16
5,null,null,null,null,null,null,0.6,0.0,0,7.2,15
6,null,null,null,null,null,null,0.615,0.0,0,null,25
7,null,null,null,null,null,null,0.6,0.0,0,5.9,10
8,4700000.00,null,null,24237430.00,null,null,1.489,6.75,6,6.2,382
9,null,null,null,null,null,null,0.6,0.0,0,7.7,25
10,6000000.00,null,null,30941400.00,null,null,13.212,6.8,15,null,236


## 3. gold.dim_genres e gold.bridge_movie_genre

A dimensão contém o catálogo único de gêneros. A bridge representa a relação muitos-para-muitos entre filmes e gêneros.


In [0]:
df_generos = spark.table("silver.tb_generos")
df_movies = spark.table("gold.dim_movies")

coluna_genero = "genero" if "genero" in df_generos.columns else "nome_genero"

df_dim_genres = (
    df_generos
    .select(F.col(coluna_genero).alias("nome_genero"))
    .filter(
        F.col("nome_genero").isNotNull() &
        (F.trim(F.col("nome_genero")) != "")
    )
    .dropDuplicates(["nome_genero"])
    .withColumn(
        "sk_genre_id",
        (F.monotonically_increasing_id() + 1).cast("bigint")
    )
    .select("sk_genre_id", "nome_genero")
)

(
    df_dim_genres.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_genres")
)

df_generos_base = (
    df_generos
    .select(
        "id_filme",
        F.col(coluna_genero).alias("nome_genero")
    )
)

df_bridge_movie_genre = (
    df_generos_base
    .join(
        df_movies.select("id_filme", "sk_movie_id"),
        on="id_filme",
        how="inner"
    )
    .join(
        df_dim_genres,
        on="nome_genero",
        how="inner"
    )
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("sk_genre_id").cast("bigint")
    )
    .dropDuplicates()
)

(
    df_bridge_movie_genre.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.bridge_movie_genre")
)

dup_genre = (
    df_bridge_movie_genre
    .groupBy("sk_movie_id", "sk_genre_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicados bridge gênero:", dup_genre)


Duplicados bridge gênero: 0


## 4. gold.dim_people, gold.dim_companies e bridges

Pessoas físicas (`Ator`, `Diretor`, `Roteirista`) ficam em `dim_people`. Produtoras ficam em `dim_companies`. As relações com filmes são representadas por tabelas-ponte.


In [0]:
df_entidades = spark.table("silver.tb_pessoas_empresas")
df_movies = spark.table("gold.dim_movies")

# -------------------------
# Dimensão de pessoas
# -------------------------
df_dim_people = (
    df_entidades
    .filter(
        F.col("tipo_entidade").isin(
            "Ator", "Diretor", "Roteirista"
        )
    )
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .dropDuplicates(["nome_pessoa", "tipo_pessoa"])
    .withColumn(
        "sk_person_id",
        (F.monotonically_increasing_id() + 1).cast("bigint")
    )
    .select(
        "sk_person_id",
        "nome_pessoa",
        "tipo_pessoa"
    )
)

(
    df_dim_people.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_people")
)

df_bridge_movie_person = (
    df_entidades
    .filter(
        F.col("tipo_entidade").isin(
            "Ator", "Diretor", "Roteirista"
        )
    )
    .select(
        "id_filme",
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .join(
        df_movies.select("id_filme", "sk_movie_id"),
        on="id_filme",
        how="inner"
    )
    .join(
        df_dim_people,
        on=["nome_pessoa", "tipo_pessoa"],
        how="inner"
    )
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("sk_person_id").cast("bigint")
    )
    .dropDuplicates()
)

(
    df_bridge_movie_person.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.bridge_movie_person")
)

# -------------------------
# Dimensão de produtoras
# -------------------------
df_dim_companies = (
    df_entidades
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(
        F.col("nome_entidade").alias("nome_produtora")
    )
    .dropDuplicates(["nome_produtora"])
    .withColumn(
        "sk_company_id",
        (F.monotonically_increasing_id() + 1).cast("bigint")
    )
    .select(
        "sk_company_id",
        "nome_produtora"
    )
)

(
    df_dim_companies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_companies")
)

df_bridge_movie_company = (
    df_entidades
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(
        "id_filme",
        F.col("nome_entidade").alias("nome_produtora")
    )
    .join(
        df_movies.select("id_filme", "sk_movie_id"),
        on="id_filme",
        how="inner"
    )
    .join(
        df_dim_companies,
        on="nome_produtora",
        how="inner"
    )
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("sk_company_id").cast("bigint")
    )
    .dropDuplicates()
)

(
    df_bridge_movie_company.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.bridge_movie_company")
)

dup_person = (
    df_bridge_movie_person
    .groupBy("sk_movie_id", "sk_person_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

dup_company = (
    df_bridge_movie_company
    .groupBy("sk_movie_id", "sk_company_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicados bridge pessoa:", dup_person)
print("Duplicados bridge empresa:", dup_company)


Duplicados bridge pessoa: 0
Duplicados bridge empresa: 0


## 5. gold.dim_reviews

As avaliações dos usuários são resumidas por filme em quantidade de avaliações e nota média arredondada em duas casas.


In [0]:
df_reviews = spark.table("silver.tb_avaliacoes_usuarios")
df_movies = spark.table("gold.dim_movies")

df_reviews_resumo = (
    df_reviews
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(
            F.avg("nota_usuario"),
            2
        ).cast("double").alias("nota_media_usuarios")
    )
)

df_dim_reviews = (
    df_reviews_resumo
    .join(
        df_movies.select("id_filme", "sk_movie_id"),
        on="id_filme",
        how="inner"
    )
    .withColumn(
        "sk_review_id",
        (F.monotonically_increasing_id() + 1).cast("bigint")
    )
    .select(
        "sk_review_id",
        F.col("sk_movie_id").cast("bigint"),
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)

(
    df_dim_reviews.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_reviews")
)

dup_review = (
    df_dim_reviews
    .groupBy("sk_review_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

dup_movie_review = (
    df_dim_reviews
    .groupBy("sk_movie_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("SK review duplicada:", dup_review)
print("Filme repetido na dim_reviews:", dup_movie_review)
display(df_dim_reviews.limit(20))


SK review duplicada: 0
Filme repetido na dim_reviews: 0


sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
1,1,1,0.1
2,5,1,3.3
3,8,1,6.8
4,17,1,5.4
5,19,1,5.3
6,20,1,4.4
7,23,3,4.75
8,24,1,4.8
9,26,1,6.6
10,33,1,7.5


## 6. gold.gold_genai_movies_context

Tabela de contexto para Vector Search/RAG. `coalesce()` fornece textos de fallback para impedir que um único campo nulo transforme todo o documento concatenado em `NULL`.


In [0]:
df_people = spark.table("gold.dim_people")
df_bridge_person = spark.table("gold.bridge_movie_person")
df_movies = spark.table("gold.dim_movies")
df_fact = spark.table("gold.fact_movies_performance")

df_movie_people = (
    df_bridge_person
    .join(df_people, on="sk_person_id", how="inner")
)

df_atores_filme = (
    df_movie_people
    .filter(F.col("tipo_pessoa") == "Ator")
    .groupBy("sk_movie_id")
    .agg(
        F.concat_ws(
            ", ",
            F.sort_array(F.collect_set("nome_pessoa"))
        ).alias("atores")
    )
)

df_diretores_filme = (
    df_movie_people
    .filter(F.col("tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(
        F.concat_ws(
            ", ",
            F.sort_array(F.collect_set("nome_pessoa"))
        ).alias("diretores")
    )
)

df_contexto_base = (
    df_movies
    .join(df_fact, on="sk_movie_id", how="inner")
    .join(df_atores_filme, on="sk_movie_id", how="left")
    .join(df_diretores_filme, on="sk_movie_id", how="left")
)

df_genai = (
    df_contexto_base
    .withColumn(
        "llm_context_document",
        F.concat(
            F.lit("O filme "),
            F.coalesce(F.col("titulo"), F.lit("título não informado")),
            F.lit(", lançado no ano de "),
            F.coalesce(
                F.col("ano_lancamento").cast("string"),
                F.lit("ano não informado")
            ),
            F.lit(", faturou R$ "),
            F.coalesce(
                F.format_number(F.col("receita_brl"), 2),
                F.lit("valor não informado")
            ),
            F.lit(" e teve um custo de R$ "),
            F.coalesce(
                F.format_number(F.col("orcamento_brl"), 2),
                F.lit("valor não informado")
            ),
            F.lit(". Estrelado por "),
            F.coalesce(F.col("atores"), F.lit("atores não informados")),
            F.lit(" e dirigido por "),
            F.coalesce(F.col("diretores"), F.lit("diretor não informado")),
            F.lit(", o filme possui a seguinte sinopse: "),
            F.coalesce(F.col("sinopse"), F.lit("sinopse não informada")),
            F.lit(".")
        )
    )
    .select(
        F.col("id_filme").alias("movie_id"),
        F.col("titulo").alias("title"),
        "llm_context_document"
    )
)

contextos_nulos = (
    df_genai
    .filter(F.col("llm_context_document").isNull())
    .count()
)

if contextos_nulos != 0:
    raise Exception(f"Foram encontrados {contextos_nulos} contextos nulos.")

(
    df_genai.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.gold_genai_movies_context")
)

print("Total de contextos:", df_genai.count())
print("Contextos nulos:", contextos_nulos)
display(df_genai.limit(10))


Total de contextos: 96463
Contextos nulos: 0


movie_id,title,llm_context_document
1000004,Purple Beatz,"O filme Purple Beatz, lançado no ano de 2022, faturou R$ valor não informado e teve um custo de R$ valor não informado. Estrelado por atores não informados e dirigido por diretor não informado, o filme possui a seguinte sinopse: Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.."
1000005,Aisha Brown: The First Black Woman Ever,"O filme Aisha Brown: The First Black Woman Ever, lançado no ano de 2020, faturou R$ valor não informado e teve um custo de R$ valor não informado. Estrelado por atores não informados e dirigido por diretor não informado, o filme possui a seguinte sinopse: No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.."
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,"O filme KYLE BROWNRIGG: INTRODUCING LYLE, lançado no ano de 2022, faturou R$ valor não informado e teve um custo de R$ valor não informado. Estrelado por atores não informados e dirigido por diretor não informado, o filme possui a seguinte sinopse: Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.."
1000011,Worth Your Weight in Gold,"O filme Worth Your Weight in Gold, lançado no ano de 2022, faturou R$ valor não informado e teve um custo de R$ valor não informado. Estrelado por atores não informados e dirigido por diretor não informado, o filme possui a seguinte sinopse: Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.."
1000014,On va manquer !,"O filme On va manquer !, lançado no ano de 2018, faturou R$ valor não informado e teve um custo de R$ valor não informado. Estrelado por atores não informados e dirigido por diretor não informado, o filme possui a seguinte sinopse: sinopse não informada."
1000030,58 Hours: The Baby Jessica Story,"O filme 58 Hours: The Baby Jessica Story, lançado no ano de 2021, faturou R$ valor não informado e teve um custo de R$ valor não informado. Estrelado por atores não informados e dirigido por diretor não informado, o filme possui a seguinte sinopse: sinopse não informada."
1000054,One Hundred Years and Hope,"O filme One Hundred Years and Hope, lançado no ano de 2022, faturou R$ valor não informado e teve um custo de R$ valor não informado. Estrelado por atores não informados e dirigido por diretor não informado, o filme possui a seguinte sinopse: In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope.."
1000058,Homecoming,"O filme Homecoming, lançado no ano de 2023, faturou R$ valor não informado e teve um custo de R$ 24,237,430.00. Estrelado por Akshay Verma, Francesca Rettondini e dirigido po

# Desafio de Analytics

As consultas abaixo respondem às seis perguntas de negócio solicitadas no projeto.


### 1. Receita total em R$


In [0]:
%sql
SELECT
    SUM(receita_brl) AS receita_total_brl
FROM gold.fact_movies_performance;


receita_total_brl
834732290730.20


### 2. Os 5 filmes com maior popularidade


In [0]:
%sql
SELECT
    m.titulo,
    f.popularidade
FROM gold.fact_movies_performance f
JOIN gold.dim_movies m
    ON f.sk_movie_id = m.sk_movie_id
WHERE f.popularidade IS NOT NULL
ORDER BY f.popularidade DESC
LIMIT 5;


titulo,popularidade
Asylum of the Devil,2.0093201020121E12
A Postcard from Pyongyang,2.0132017E7
"The psychopath, chronicle of an unsolved case",1.9851995E7
Russian Hackers: The Beginning,902010.0
Mississippi Madam: The Life of Nellie Jackson,601990.0


### 3. Quantidade de filmes por gênero


In [0]:
%sql
SELECT
    g.nome_genero,
    COUNT(DISTINCT b.sk_movie_id) AS quantidade_filmes
FROM gold.bridge_movie_genre b
JOIN gold.dim_genres g
    ON b.sk_genre_id = g.sk_genre_id
GROUP BY g.nome_genero
ORDER BY quantidade_filmes DESC;


nome_genero,quantidade_filmes
Crime|drama|romance,30457
Adventure|family,18612
Horror,17348
"""and I Miss Her Already\""""\""""""""drama",9424
/u0xxpis5tasfqtlvljocx56kpyu.jpg,9163
Horror|drama|mystery|thriller,6996
Animation|family|fantasy|comedy|adventure|mystery,5541
"Violence And Broken Lives.""",4320
/ialjmnflmgnk4xtgkxc3ecfjwd9.jpg,4159
Crime|tv Movie|mystery|drama,3676


### 4. Top 10 filmes por receita, com RANK()


In [0]:
%sql
WITH ranking AS (
    SELECT
        m.titulo,
        f.receita_usd,
        f.receita_brl,
        RANK() OVER (
            ORDER BY f.receita_usd DESC
        ) AS posicao
    FROM gold.fact_movies_performance f
    JOIN gold.dim_movies m
        ON f.sk_movie_id = m.sk_movie_id
    WHERE f.receita_usd IS NOT NULL
)
SELECT *
FROM ranking
ORDER BY posicao
LIMIT 10;


titulo,receita_usd,receita_brl,posicao
Avengers: Endgame,2800000000.00,14439320000.00,1
Avatar: The Way of Water,2320250281.00,11965298674.09,2
AVENGERS: INFINITY WAR,2052415039.00,10584099114.62,3
spider-man: no way home,1921847111.00,9910773366.72,4
The Lion King,1663075401.00,8576313535.42,5
Top Gun: Maverick,1488732821.00,7677246284.61,6
Barbie,1428545028.00,7366863854.89,7
The Super Mario Bros. Movie,1355725263.00,6991339608.76,8
Black Panther,1349926083.00,6961433817.42,9
Star Wars: The Last Jedi,1332698830.00,6872594596.43,10


### 5. Ator com maior número de participações nos últimos 2 anos

O limite superior é a data de lançamento válida mais recente da base, ignorando datas futuras e filmes não lançados.


In [0]:
%sql
WITH data_limite AS (
    SELECT MAX(data_lancamento) AS data_maxima
    FROM gold.dim_movies
    WHERE status_filme = 'Lançado'
      AND data_lancamento <= CURRENT_DATE()
),
participacoes AS (
    SELECT
        p.nome_pessoa,
        COUNT(DISTINCT m.sk_movie_id) AS quantidade_filmes
    FROM gold.bridge_movie_person b
    JOIN gold.dim_people p
        ON b.sk_person_id = p.sk_person_id
    JOIN gold.dim_movies m
        ON b.sk_movie_id = m.sk_movie_id
    CROSS JOIN data_limite d
    WHERE p.tipo_pessoa = 'Ator'
      AND m.status_filme = 'Lançado'
      AND m.data_lancamento BETWEEN ADD_MONTHS(d.data_maxima, -24)
                                AND d.data_maxima
    GROUP BY p.nome_pessoa
)
SELECT *
FROM participacoes
ORDER BY quantidade_filmes DESC
LIMIT 1;


nome_pessoa,quantidade_filmes
Colette Turner,56


### 6. Produtora com maior lucro nos últimos 5 anos

Para tornar a métrica explícita, o lucro é agregado em BRL. O limite superior segue a mesma regra da data de lançamento válida mais recente.


In [0]:
%sql
WITH data_limite AS (
    SELECT MAX(data_lancamento) AS data_maxima
    FROM gold.dim_movies
    WHERE status_filme = 'Lançado'
      AND data_lancamento <= CURRENT_DATE()
),
lucro_produtoras AS (
    SELECT
        c.nome_produtora,
        SUM(f.lucro_brl) AS lucro_total_brl
    FROM gold.bridge_movie_company b
    JOIN gold.dim_companies c
        ON b.sk_company_id = c.sk_company_id
    JOIN gold.dim_movies m
        ON b.sk_movie_id = m.sk_movie_id
    JOIN gold.fact_movies_performance f
        ON b.sk_movie_id = f.sk_movie_id
    CROSS JOIN data_limite d
    WHERE m.status_filme = 'Lançado'
      AND m.data_lancamento BETWEEN ADD_MONTHS(d.data_maxima, -60)
                                AND d.data_maxima
      AND f.lucro_brl IS NOT NULL
    GROUP BY c.nome_produtora
)
SELECT *
FROM lucro_produtoras
ORDER BY lucro_total_brl DESC
LIMIT 1;


nome_produtora,lucro_total_brl
Type Investigations,29966650615.96


## Validação final da camada Gold


In [0]:
tabelas_gold = [
    "gold.dim_movies",
    "gold.fact_movies_performance",
    "gold.dim_genres",
    "gold.dim_people",
    "gold.dim_companies",
    "gold.dim_reviews",
    "gold.bridge_movie_genre",
    "gold.bridge_movie_person",
    "gold.bridge_movie_company",
    "gold.gold_genai_movies_context"
]

for tabela in tabelas_gold:
    print(f"{tabela}: {spark.table(tabela).count()} linhas")

print("Duplicados na fato:", duplicados_fact)
print("Duplicados bridge gênero:", dup_genre)
print("Duplicados bridge pessoa:", dup_person)
print("Duplicados bridge empresa:", dup_company)
print("SK review duplicada:", dup_review)
print("Filme repetido na dim_reviews:", dup_movie_review)
print("Contextos nulos:", contextos_nulos)


gold.dim_movies: 97879 linhas
gold.fact_movies_performance: 96463 linhas
gold.dim_genres: 3490 linhas
gold.dim_people: 420863 linhas
gold.dim_companies: 45946 linhas
gold.dim_reviews: 27303 linhas
gold.bridge_movie_genre: 137773 linhas
gold.bridge_movie_person: 785212 linhas
gold.bridge_movie_company: 118893 linhas
gold.gold_genai_movies_context: 96463 linhas
Duplicados na fato: 0
Duplicados bridge gênero: 0
Duplicados bridge pessoa: 0
Duplicados bridge empresa: 0
SK review duplicada: 0
Filme repetido na dim_reviews: 0
Contextos nulos: 0
